# 05 — TeleOCR LoRA Fine-tuning — Experimental Adaptation (Google Colab Pro)

**Important status:** the TeleOCR model card currently provides an official inference recipe, but no official downstream LoRA/SFT recipe comparable to GLM-OCR. This notebook is therefore an **experimental PEFT adaptation**, grounded in the official model input template plus Hugging Face PEFT/Trainer conventions.

Safety design:
- target only language-model linear projection layers; vision modules must remain untouched;
- micro-batch is fixed to 1 for robust variable-image handling, with gradient accumulation 16;
- perform a one-sample forward/loss gate before any training;
- run a 128/32 one-epoch smoke job first;
- save adapter-only epoch checkpoints to Drive;
- final checkpoint selection is done later by validation CER, not training loss.

## 0. Install

In [ ]:
%pip install -q -U "kagglehub>=1.0.2" "jiwer>=4.0.0" pandas pillow
%pip install -q -U "transformers>=4.57.1,<5.0" "accelerate>=1.2.0" "peft>=0.15.0" datasets huggingface_hub

## 1. Data

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

## 2. GPU and hyperparameters

In [ ]:
import torch
assert torch.cuda.is_available(),'GPU required.'
torch.manual_seed(SEED)
MODEL_ID='StarDoc-AI/TeleOCR'; SYSTEM_PROMPT='You are a helpful assistant.'; OCR_PROMPT='Please output the text content from the image.'
GPU_NAME=torch.cuda.get_device_name(0); VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3; BF16=torch.cuda.is_bf16_supported(); DTYPE=torch.bfloat16 if BF16 else torch.float16
MICRO_BATCH=1; GRAD_ACC=16; LR=1e-4; EPOCHS=3
OUTPUT_DIR=PROJECT_ROOT/'checkpoints'/'teleocr_lora'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print(f'GPU={GPU_NAME} | VRAM={VRAM_GB:.1f}GB | BF16={BF16} | effective batch={MICRO_BATCH*GRAD_ACC}')

## 3. Load base model and processor

In [ ]:
from transformers import AutoProcessor
try:
    from transformers import AutoModelForMultimodalLM
    ModelClass=AutoModelForMultimodalLM
except ImportError:
    from transformers import AutoModel
    ModelClass=AutoModel
processor=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=True,use_fast=True)
base_model=ModelClass.from_pretrained(MODEL_ID,trust_remote_code=True,torch_dtype=DTYPE)
base_model.config.use_cache=False
try: base_model.gradient_checkpointing_enable()
except Exception as e: print('gradient_checkpointing_enable not exposed:',repr(e))
print('Base class:',base_model.__class__.__name__)

## 4. Discover PEFT targets — language model only

In [ ]:
TARGET_SUFFIXES={'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'}
linear_names=[name for name,module in base_model.named_modules() if isinstance(module,torch.nn.Linear)]
target_names=[name for name,module in base_model.named_modules() if isinstance(module,torch.nn.Linear) and 'language_model' in name.lower() and name.split('.')[-1] in TARGET_SUFFIXES]
print('all linear modules:',len(linear_names)); print('selected targets:',len(target_names)); print('\n'.join(target_names[:30]))
if not target_names:
    print('First 80 linear module names for debugging:'); print('\n'.join(linear_names[:80]))
    raise RuntimeError('No language_model LoRA targets discovered. Stop here rather than targeting vision layers accidentally.')
assert all('vision' not in n.lower() for n in target_names)

## 5. Inject LoRA

In [ ]:
from peft import LoraConfig,get_peft_model
lora_cfg=LoraConfig(r=8,lora_alpha=32,lora_dropout=0.05,bias='none',target_modules=target_names)
model=get_peft_model(base_model,lora_cfg)
model.print_trainable_parameters()
# Strong guard: every trainable parameter must belong to LoRA.
trainable=[n for n,p in model.named_parameters() if p.requires_grad]
assert trainable and all('lora_' in n.lower() for n in trainable),trainable[:20]

## 6. Dataset and micro-batch-1 multimodal collator

In [ ]:
from torch.utils.data import Dataset
class RowDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,idx): return {'idx':int(idx)}

class TeleOCRCollator:
    def __init__(self,frame,processor): self.frame=frame.reset_index(drop=True); self.processor=processor
    def __call__(self,features):
        if len(features)!=1: raise ValueError('This robust collator intentionally requires micro-batch=1.')
        row=self.frame.iloc[int(features[0]['idx'])]; image=Image.open(resolve_image_path(row)).convert('RGB'); gt=str(row['text'])
        prompt_messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':[{'type':'image'},{'type':'text','text':OCR_PROMPT}]}]
        full_messages=prompt_messages+[{'role':'assistant','content':gt}]
        prompt_text=self.processor.apply_chat_template(prompt_messages,tokenize=False,add_generation_prompt=True)
        full_text=self.processor.apply_chat_template(full_messages,tokenize=False,add_generation_prompt=False)
        full=self.processor(text=[full_text],images=[image],padding=True,return_tensors='pt')
        prompt=self.processor(text=[prompt_text],images=[image],padding=True,return_tensors='pt')
        labels=full['input_ids'].clone(); prompt_len=prompt['input_ids'].shape[1]; labels[:,:prompt_len]=-100
        pad_id=getattr(self.processor.tokenizer,'pad_token_id',None)
        if pad_id is not None: labels[full['input_ids']==pad_id]=-100
        full['labels']=labels
        return full

## 7. Mandatory one-sample forward/loss gate

In [ ]:
collator=TeleOCRCollator(train_df,processor); sample_batch=collator([{'idx':0}])
model=model.cuda()
move={k:(v.cuda() if torch.is_tensor(v) else v) for k,v in sample_batch.items()}
with torch.no_grad(): out=model(**move)
loss=getattr(out,'loss',None)
print('loss=',loss)
assert loss is not None and torch.isfinite(loss).item(),'TeleOCR custom model did not return a finite supervised loss. Do not launch training.'
del move,out; torch.cuda.empty_cache()
print('✅ Forward/loss gate passed.')

## 8. Trainer builder

In [ ]:
from transformers import Trainer,TrainingArguments

def make_trainer(model,train_frame,val_frame,output_dir,epochs):
    args=TrainingArguments(
        output_dir=str(output_dir),num_train_epochs=epochs,
        per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR,lr_scheduler_type='cosine',warmup_ratio=0.1,
        eval_strategy='epoch',save_strategy='epoch',logging_steps=10,save_total_limit=3,save_only_model=True,
        bf16=BF16,fp16=not BF16,gradient_checkpointing=True,remove_unused_columns=False,report_to='none',seed=SEED,
    )
    return Trainer(model=model,args=args,train_dataset=RowDataset(train_frame),eval_dataset=RowDataset(val_frame),data_collator=TeleOCRCollator(train_frame,processor))

### Collator note
For evaluation, Hugging Face `Trainer` uses the same data collator object. Therefore the validation collator must point at the validation frame. We override `get_eval_dataloader` below to keep the frame association correct.

In [ ]:
from torch.utils.data import DataLoader
class FrameAwareTrainer(Trainer):
    def __init__(self,*args,train_frame=None,val_frame=None,processor=None,**kwargs):
        super().__init__(*args,**kwargs); self._train_frame=train_frame; self._val_frame=val_frame; self._processor=processor
    def get_train_dataloader(self):
        return DataLoader(self.train_dataset,batch_size=1,shuffle=True,collate_fn=TeleOCRCollator(self._train_frame,self._processor),num_workers=0)
    def get_eval_dataloader(self,eval_dataset=None):
        ds=eval_dataset if eval_dataset is not None else self.eval_dataset
        return DataLoader(ds,batch_size=1,shuffle=False,collate_fn=TeleOCRCollator(self._val_frame,self._processor),num_workers=0)

def make_trainer(model,train_frame,val_frame,output_dir,epochs):
    args=TrainingArguments(output_dir=str(output_dir),num_train_epochs=epochs,per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=GRAD_ACC,learning_rate=LR,lr_scheduler_type='cosine',warmup_ratio=0.1,eval_strategy='epoch',save_strategy='epoch',logging_steps=10,save_total_limit=3,save_only_model=True,bf16=BF16,fp16=not BF16,gradient_checkpointing=True,remove_unused_columns=False,report_to='none',seed=SEED)
    return FrameAwareTrainer(model=model,args=args,train_dataset=RowDataset(train_frame),eval_dataset=RowDataset(val_frame),data_collator=TeleOCRCollator(train_frame,processor),train_frame=train_frame,val_frame=val_frame,processor=processor)

## 9. One-epoch 128/32 smoke training

In [ ]:
RUN_SMOKE_TRAINING=True
SMOKE_DIR=Path('/content/teleocr_lora_smoke'); smoke_train=train_df.sample(n=128,random_state=SEED).reset_index(drop=True); smoke_val=val_df.sample(n=32,random_state=SEED).reset_index(drop=True)
if RUN_SMOKE_TRAINING:
    trainer=make_trainer(model,smoke_train,smoke_val,SMOKE_DIR,1); result=trainer.train(); print(result)
    assert list(SMOKE_DIR.glob('checkpoint-*')) or (SMOKE_DIR/'adapter_config.json').exists(); print('✅ Smoke training/save passed.')
else: print('Smoke skipped.')

## 10. Rebuild a clean model for the full run

In [ ]:
def build_clean_lora_model():
    base=ModelClass.from_pretrained(MODEL_ID,trust_remote_code=True,torch_dtype=DTYPE)
    base.config.use_cache=False
    try: base.gradient_checkpointing_enable()
    except Exception: pass
    m=get_peft_model(base,lora_cfg); return m

## 11. Full 3-epoch run — gated

In [ ]:
RUN_FULL_TRAINING=False
if RUN_FULL_TRAINING:
    del trainer,model; gc.collect(); torch.cuda.empty_cache()
    full_model=build_clean_lora_model(); full_trainer=make_trainer(full_model,train_df,val_df,OUTPUT_DIR,EPOCHS); result=full_trainer.train(); print(result)
    print('Saved epoch adapters under',OUTPUT_DIR)
else: print('Full training gated. Set RUN_FULL_TRAINING=True only after smoke gate passes.')

## 12. Save experiment provenance

In [ ]:
prov={'model_id':MODEL_ID,'status':'EXPERIMENTAL downstream LoRA adaptation; no official TeleOCR fine-tuning recipe located','lora_rank':8,'lora_alpha':32,'lora_dropout':0.05,'target_modules':target_names,'learning_rate':LR,'epochs':EPOCHS,'micro_batch':1,'gradient_accumulation':GRAD_ACC,'effective_batch':GRAD_ACC,'gpu':GPU_NAME,'vram_gb':VRAM_GB,'bf16':BF16,'train_samples':len(train_df),'val_samples':len(val_df),'seed':SEED}
(OUTPUT_DIR.parent/'teleocr_training_provenance.json').write_text(json.dumps(prov,ensure_ascii=False,indent=2),encoding='utf-8')